In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from google.cloud import storage
from collections import Counter

import pandas as pd
import numpy as np
import string
import time
import copy
import random
import json
import requests
import os
import hashlib
import warnings


import preprocess
import model_arch
import update_model

In [2]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

DEVICE = get_device()
print('Using device:', str(DEVICE).upper(), "\n")

MODEL_DIR="./model"
DATA_PATH = "war_and_peace.txt"

TRAIN_DATA_BUCKET = "cbow-training-data-1f656"
MODEL_DATA_BUCKET = "cbow-model-data-1f656"

MPS is available
Using device: MPS 



In [8]:
# === Run this code for the first model initialization ===

# # Download training data
# preprocess.download_data_from_gcs(
#     TRAIN_DATA_BUCKET,
#     DATA_PATH,
# )

# model, cfg = update_model.init_first_model(data_path=DATA_PATH)

In [3]:
# === Run this code to initialize pretrained model ===

# Download training data
preprocess.download_data_from_gcs(
    TRAIN_DATA_BUCKET,
    DATA_PATH
)

# Download pretrained model and its configuration
preprocess.download_model_from_gcs(
    bucket_name=MODEL_DATA_BUCKET,
    download_dir=MODEL_DIR,
)

# Initialize pretrained model
model, cfg = update_model.init_model(
    new_data_path=f"data/{DATA_PATH}"
)

Data already exists at data/war_and_peace.txt
Latest version folder: version-1
Model already exists at ./model/model_cpu.pt
Model already exists at ./model/model_ns_config.json
Model already exists at ./model/sequence.pt
 CBOW Incremental Update
   new data : data/war_and_peace.txt
   device   : MPS
 -- Loaded weights from ./model/model_cpu.pt
 -- Skipping 'data/war_and_peace.txt': already ingested (hash match)
 -- Vocab unchanged; no layer resizing needed

 -- Done ✓


In [10]:
# === Train model ===
model = update_model.train(
    model=model,
    cfg=cfg,
    device=DEVICE,
    epochs=2,
)


 -- Training for 2 epoch(s) on 5,030,471 samples (NEG k=5, device=MPS)

   Epoch   1/2  loss=6463.1026  time=52.8s
   Epoch   2/2  loss=5983.1482  time=50.9s


In [4]:
# === Save model configuration locally ===
update_model.save(model, cfg)


 -- Weights  saved → ./model/model_cpu.pt
 -- Config   saved → ./model/model_ns_config.json


In [5]:
# === Save model configuration in GCS and create new version ===
preprocess.save_model_to_gcs(bucket_name=MODEL_DATA_BUCKET)

version-2
Uploaded ./model/model_ns_config.json → gs://cbow-model-data-1f656/version-2/model_ns_config.json
Uploaded ./model/sequence.pt → gs://cbow-model-data-1f656/version-2/sequence.pt
Uploaded ./model/model_cpu.pt → gs://cbow-model-data-1f656/version-2/model_cpu.pt


In [6]:
window_size = cfg['window_size']
word2idx    = cfg['word2idx']

In [7]:
model = model.to(DEVICE)
word_embeddings = model.embedding.weight.detach()

# Example prediction
sentense = "i want to break free".lower()
context_words = []

split_sentense = sentense.split(" ")

if len(split_sentense) != window_size * 2 + 1:
    print("ERROR: sentense does not have the сorrect lenght for testinng")
else:
    for w in split_sentense:
        context_words.append(w)
    del context_words[window_size]

print("Context words: ", context_words)

Context words:  ['i', 'want', 'break', 'free']


In [9]:
top_k = 5
model = model.to(DEVICE)
model.eval()

idx2word = {}
for k, v in word2idx.items():
    idx2word[v] = k

context_idxs = torch.tensor(
    [[word2idx[w] for w in context_words]]
)
context_idxs = context_idxs.to(DEVICE)

with torch.no_grad():
    logits = model(context_idxs)
    probs = torch.softmax(logits, dim=1)

top_idxs = torch.topk(probs, top_k).indices[0]
top_idxs
result = [idx2word[idx.item()] for idx in top_idxs]
print(result)

TypeError: CBOWModelNS.forward() missing 2 required positional arguments: 'target' and 'neg_samples'

In [14]:
word_1 = "white"
word_2 = "green"

v1 = word_embeddings[word2idx[word_1]]
v2 = word_embeddings[word2idx[word_2]]
print(f"Original: {F.cosine_similarity(v1, v2, dim=0).item()}")

Original: 0.4399053752422333


In [9]:
# coincides = 0
# w1_pos = 0
# w2_pos = 0
# for i in range(len(word_emb1)):
#     if word_emb1[i] == 1.0:
#         w1_pos += 1
#     if word_emb2[i] == 1.0:
#         w2_pos += 1
#     if word_emb1[i] == 1.0 and word_emb2[i] == 1.0:
#         coincides += 1

# print(f"Number of 1-position for {word_1}: {w1_pos}")
# print(f"Number of 1-position for {word_2}: {w2_pos}")
# print(f"Number of coincides: {coincides}")

In [10]:
words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

In [12]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

# thresholds = [0.1,0.2,0.3,0.4]
thresholds = [i for i in range(1, 11)]

for t in thresholds:
    # filtering
    B = (embeddings >= t).int()
    # C = torch.cov(B.T)
    C = B @ B.T

    print(f" -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_1-cov_matrix_{t}.csv")

 -- threshold = 1, nonzoer(B) = 1088
 -- threshold = 2, nonzoer(B) = 774
 -- threshold = 3, nonzoer(B) = 568
 -- threshold = 4, nonzoer(B) = 408
 -- threshold = 5, nonzoer(B) = 296
 -- threshold = 6, nonzoer(B) = 220
 -- threshold = 7, nonzoer(B) = 162
 -- threshold = 8, nonzoer(B) = 115
 -- threshold = 9, nonzoer(B) = 78
 -- threshold = 10, nonzoer(B) = 59


In [13]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

deltas = [1, 2, 3] # [0.02,0.05]

for d in deltas:
    # filtering
    B = (torch.abs(embeddings) <= d).int()
    C = B @ B.T

    print(f"-- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_2-cov_matrix_{d}.csv")

-- delta = 1, nonzoer(B) = 796
-- delta = 2, nonzoer(B) = 1540
-- delta = 3, nonzoer(B) = 2103
